In [1]:
from langchain_openai import ChatOpenAI  
from langchain_core.messages import HumanMessage

LLM = ChatOpenAI(model="gpt-4o-mini")                 # Initialize the chat model

LLM.invoke([HumanMessage(content="안녕, 잘 지냈어?")]) # Send a message to the model

AIMessage(content='안녕하세요! 저는 잘 지내고 있어요. 당신은 어떻게 지내셨나요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 15, 'total_tokens': 34, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_af5b2e4445', 'id': 'chatcmpl-Eg1lsVCMDDrhyhc5rcZrN1YewdsOU', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a19ff9-8238-7501-b73e-c3fa31619d46-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 19, 'total_tokens': 34, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [2]:
from langchain_core.tools import tool
from datetime import datetime
import pytz

@tool       # @tool decorator uses langchain's tool system
def get_current_time(timezone: str, location: str) -> str:
    """ 현재 시각 반환 함수
    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 LLM 답변 생성에 사용됨
    """
    Time_zone = pytz.timezone(timezone)
    now = datetime.now(Time_zone).strftime("%Y-%m-%d %H:%M:%S")

    # Return format: 'Asia/Seoul (Seoul) 현재시각 2024-06-01 15:30:00'
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} '
     
    print(location_and_local_time)
    return location_and_local_time

In [3]:
# Append the tool to a list and a dictionary
tools = [get_current_time,]
tool_dict = {"get_current_time": get_current_time,}

# Bind the tools to the LLM 
LLM_with_tools = LLM.bind_tools(tools)

In [4]:
from langchain_core.messages import SystemMessage

# Create messages for the conversation
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools 사용 가능한 AI 어시스턴트야."),
    HumanMessage("인천은 지금 몇시야?"),
]

# Call the LLM_with_tools to generate a response
response = LLM_with_tools.invoke(messages)
messages.append(response)

# Print the conversation messages
print(messages)

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools 사용 가능한 AI 어시스턴트야.', additional_kwargs={}, response_metadata={}), HumanMessage(content='인천은 지금 몇시야?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 138, 'total_tokens': 161, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_fbcfd2e298', 'id': 'chatcmpl-Eg1m0hAA3Ofz1C8VUonoux0n0GCHh', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a19ff9-a28c-71b3-b750-efb5f30969ca-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezo

In [5]:
for tool_call in response.tool_calls:
    # Select the tool in tool_dict based on the tool name
    selected_tool = tool_dict[tool_call["name"]] 
    print(tool_call["args"])                   # Arguments for the tool function
    tool_msg = selected_tool.invoke(tool_call) # Result from the tool function
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Seoul', 'location': '인천'}
Asia/Seoul (인천) 현재시각 2026-11-03 14:55:06 


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools 사용 가능한 AI 어시스턴트야.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='인천은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 138, 'total_tokens': 161, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_fbcfd2e298', 'id': 'chatcmpl-Eg1m0hAA3Ofz1C8VUonoux0n0GCHh', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a19ff9-a28c-71b3-b750-efb5f30969ca-0', tool_calls=[{'name': 'get_current_time', 'args': {'time

In [8]:
LLM_with_tools.invoke(messages)

AIMessage(content='인천 현재 시각은 2025년 12월 4일 13시 48분 38초입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 195, 'total_tokens': 223, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_b547601dbd', 'id': 'chatcmpl-CivZ2K8c4pmvOQ9nzzwaFq4ujCEFZ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--e9a3fc04-4963-4c9c-a30d-d7fae8248553-0', usage_metadata={'input_tokens': 195, 'output_tokens': 28, 'total_tokens': 223, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})